# Resumen Experimento 2: Beta=0 vs Beta optima, Observacional vs Intervencional

Combina, en una unica tabla indexada por `(Ruido, N)`, los resultados ya calculados en
`Resumen_Sinteticos_Observacional.ipynb` y `Resumen_Sinteticos_Intervencional.ipynb` (Experimento 3
sobre los 11 datasets sinteticos): para cada metrica (`RF Acc`, `MMD`) y cada experimento (`obs`,
`int`) se muestra el valor medio en `Beta=0` (baseline) y en la `Beta` optima. La `beta_optima` es la
misma para ambos experimentos, ya que `Resumen_Sinteticos_Intervencional.ipynb` la importa de
`Resumen_Sinteticos_Observacional.ipynb` en lugar de re-seleccionarla.

Es la misma tabla `comparacion` de esos dos notebooks, pero:
- sin las columnas `n_pares` y `Cobertura`.
- sustituyendo los p-valores de Wilcoxon por los valores medios en `Beta=0` y `Beta=beta_optima` (las
  columnas `RF Acc beta=0` / `MMD beta=0` que ya existian en las tablas por N, pero no en `comparacion`).
- con las columnas agrupadas por experimento (`Observacional` antes que `Intervencional`) y, dentro de
  cada experimento, `RF Acc` antes que `MMD`; `beta_optima` se mantiene en su misma posicion original
  (primera columna).

No se recalcula nada: los datos se importan directamente de los CSV resumen ya guardados por ambos
notebooks en `notebooks/tablas/`.

In [33]:
import pandas as pd
from pathlib import Path

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
TABLAS_DIR = NOTEBOOKS_DIR / "tablas"

ALPHA = 0.05

## Fuentes de datos

Los 4 CSV resumen ya calculados por `Resumen_Sinteticos_Observacional.ipynb` (Experimento 3
observacional) y `Resumen_Sinteticos_Intervencional.ipynb` (Experimento 3 bajo `do(x2=0)`), uno por
combinacion (Experimento, Ruido).

In [34]:
FUENTES = {
    ("Observacional", "Gaussiano"): TABLAS_DIR / "resumen_sinteticos_observacional_beta_gaussiano.csv",
    ("Observacional", "Gamma"): TABLAS_DIR / "resumen_sinteticos_observacional_beta_gamma.csv",
    ("Intervencional", "Gaussiano"): TABLAS_DIR / "resumen_sinteticos_intervencional_beta_gaussiano.csv",
    ("Intervencional", "Gamma"): TABLAS_DIR / "resumen_sinteticos_intervencional_beta_gamma.csv",
}

for path in FUENTES.values():
    assert path.exists(), f"No existe: {path}"

tablas = {clave: pd.read_csv(path).set_index("N") for clave, path in FUENTES.items()}

## Combinar Gaussiano + Gamma por experimento

Igual que la celda `comparacion` de los notebooks originales: concatena Gaussiano y Gamma en el eje
`Ruido`, indexado por `(Ruido, N)`. Se comprueba tambien que `beta_optima` coincide entre Observacional
e Intervencional, ya que este ultimo la importa del primero en vez de re-seleccionarla.

In [35]:
def combinar_ruidos(experimento: str) -> pd.DataFrame:
    return pd.concat(
        {ruido: tablas[(experimento, ruido)] for ruido in ("Gaussiano", "Gamma")},
        names=["Ruido", "N"],
    )


obs = combinar_ruidos("Observacional")
intv = combinar_ruidos("Intervencional")

assert obs["beta_optima"].equals(intv["beta_optima"]), \
    "beta_optima difiere entre Observacional e Intervencional: Intervencional deberia importarla de Observacional"

## Construir la tabla comparacion

Para cada metrica (`RF Acc`, `MMD`) y cada experimento (`obs`, `int`), se toman los valores medios en
`Beta=0` y `Beta=beta_optima` ya calculados por N en las tablas fuente. Orden de columnas: `beta_optima`
en su posicion original, luego `Observacional` antes que `Intervencional`, y dentro de cada experimento
`RF Acc` antes que `MMD`.

In [36]:
def con_sufijo(df: pd.DataFrame, sufijo: str) -> pd.DataFrame:
    """Selecciona RF Acc/MMD en Beta=0 y Beta=beta_optima, renombrando con el sufijo del experimento."""
    columnas = ["RF Acc beta=0", "RF Acc beta_optima", "MMD beta=0", "MMD beta_optima"]
    return df[columnas].rename(columns={c: f"{c.rsplit(' ', 1)[0]} {sufijo} {c.rsplit(' ', 1)[1]}" for c in columnas})


comparacion = pd.concat([obs[["beta_optima"]], con_sufijo(obs, "obs"), con_sufijo(intv, "int")], axis=1)

comparacion = comparacion[[
    "beta_optima",
    "RF Acc obs beta=0", "RF Acc obs beta_optima", "MMD obs beta=0", "MMD obs beta_optima",
    "RF Acc int beta=0", "RF Acc int beta_optima", "MMD int beta=0", "MMD int beta_optima",
]]

## Desviacion estandar por metrica

Las tablas resumen (`tablas`) solo tienen la media por (Ruido, N, Beta): para la std hace falta el CSV
crudo por (Dataset, Seed), el mismo que usan `Resumen_Sinteticos_Observacional.ipynb` /
`Resumen_Sinteticos_Intervencional.ipynb` antes de agregar (`Experimento3 datasets/*/tablas/datasets_sinteticos_*.csv`).
Se calcula la std entre `(Dataset, Seed)` en `Beta=0` y `Beta=beta_optima`, agrupando igual que las
medias de `comparacion`.

In [37]:
FUENTES_RAW = {
    ("Observacional", "Gaussiano"): NOTEBOOKS_DIR / "Experimento3 datasets" / "Obervacional" / "tablas" / "datasets_sinteticos_observacional.csv",
    ("Observacional", "Gamma"): NOTEBOOKS_DIR / "Experimento3 datasets" / "Obervacional" / "tablas" / "datasets_sinteticos_observacional_gamma.csv",
    ("Intervencional", "Gaussiano"): NOTEBOOKS_DIR / "Experimento3 datasets" / "Intervencional" / "tablas" / "datasets_sinteticos_intervencional.csv",
    ("Intervencional", "Gamma"): NOTEBOOKS_DIR / "Experimento3 datasets" / "Intervencional" / "tablas" / "datasets_sinteticos_intervencional_gamma.csv",
}

for path in FUENTES_RAW.values():
    assert path.exists(), f"No existe: {path}"

raw = {clave: pd.read_csv(path) for clave, path in FUENTES_RAW.items()}


def std_por_beta(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=("RF Acc", "MMD")):
    """Std entre (Dataset, Seed) en Beta=0 y Beta=beta_opt, agrupando igual que las medias de `comparacion`."""
    df_n = df[df["N"] == n_filter]
    std_por_n = df_n.groupby("Beta")[list(metrics)].std()
    return std_por_n.loc[0.0], std_por_n.loc[beta_opt]


def stds_de(experimento: str) -> pd.DataFrame:
    filas = {}
    for ruido in ("Gaussiano", "Gamma"):
        for n_filter in (50, 100):
            beta_opt = tablas[(experimento, ruido)].loc[n_filter, "beta_optima"]
            std0, std_opt = std_por_beta(raw[(experimento, ruido)], beta_opt, n_filter)
            filas[(ruido, n_filter)] = {
                "RF Acc beta=0": std0["RF Acc"], "RF Acc beta_optima": std_opt["RF Acc"],
                "MMD beta=0": std0["MMD"], "MMD beta_optima": std_opt["MMD"],
            }
    resultado = pd.DataFrame.from_dict(filas, orient="index")
    resultado.index = pd.MultiIndex.from_tuples(resultado.index, names=["Ruido", "N"])
    return resultado


stds = pd.concat([con_sufijo(stds_de("Observacional"), "obs"), con_sufijo(stds_de("Intervencional"), "int")], axis=1)
stds = stds[[c for c in comparacion.columns if c != "beta_optima"]]

## Formato

Cada metrica se muestra como `media<sub>std</sub>` (2 decimales cada una), combinando `comparacion`
(medias) con `stds` (desviaciones). `beta_optima` se muestra con 1 decimal y sin std, ya que es un
valor elegido, no un promedio.

In [38]:
def celda_con_std(media: float, std: float) -> str:
    return f"{media:.2f}<sub>{std:.2f}</sub>"


def con_std(medias: pd.DataFrame, stds: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=medias.index)
    out["beta_optima"] = medias["beta_optima"].map(lambda b: f"{b:.1f}")
    for col in stds.columns:
        out[col] = [celda_con_std(m, s) for m, s in zip(medias[col], stds[col])]
    return out

## Resaltado de celdas significativas

Igual que en `Resumen_Sinteticos_Observacional.ipynb` / `Resumen_Sinteticos_Intervencional.ipynb`: las
celdas `*_optima` se muestran en negrita cuando su p-valor de Wilcoxon (Beta=0 vs Beta=beta_optima,
ya calculado en las tablas fuente) es significativo. El p-valor en si no se anade como columna, solo
se usa para el estilo visual sobre la celda `media<sub>std</sub>`.

In [39]:
def p_valores_de(df: pd.DataFrame, sufijo: str) -> pd.DataFrame:
    return df[["RF Acc p_valor", "MMD p_valor"]].rename(columns={
        "RF Acc p_valor": f"RF Acc {sufijo} p_valor",
        "MMD p_valor": f"MMD {sufijo} p_valor",
    })


p_valores = pd.concat([p_valores_de(obs, "obs"), p_valores_de(intv, "int")], axis=1)

COLUMNA_A_PVALOR = {
    "RF Acc obs beta_optima": "RF Acc obs p_valor",
    "RF Acc int beta_optima": "RF Acc int p_valor",
    "MMD obs beta_optima": "MMD obs p_valor",
    "MMD int beta_optima": "MMD int p_valor",
}


def tabla_con_negrita(medias: pd.DataFrame, stds: pd.DataFrame):
    tabla = con_std(medias, stds)

    def resaltar(row):
        estilos = []
        for col in row.index:
            col_p = COLUMNA_A_PVALOR.get(col)
            p_val = p_valores.loc[row.name, col_p] if col_p is not None else None
            estilos.append("font-weight: bold" if pd.notna(p_val) and p_val < ALPHA else "")
        return estilos

    return tabla.style.apply(resaltar, axis=1)

## Tabla final

In [40]:
tabla_con_negrita(comparacion, stds)

## Guardar CSV resumen

In [41]:
OUT_PATH = TABLAS_DIR / "resumen_experimento2_tfm.csv"
comparacion.to_csv(OUT_PATH)

print(f"Guardado en: {OUT_PATH}")

Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\tablas\resumen_experimento2_tfm.csv
